# 04 — Entity Extraction and Candidate Dictionary

**Project:** From Online Attention to Electoral Outcomes: A Network Science Analysis of Philippine Election 2025 Twitter/X Communication

This notebook is part of the notebook set for the Philippine Election 2025 Twitter/X network science project.



## Visible progress tracker

This notebook now prints numbered stage updates while it runs. If Jupyter shows `[*]`, the current stage is still processing. A run log is also saved under `outputs/run_logs/`.


In [ ]:
# VISIBLE PROGRESS TRACKER — automatically added in v5
from pathlib import Path as _ProgressPath
import sys as _progress_sys
_PROGRESS_ROOT = _ProgressPath.cwd().parent if _ProgressPath.cwd().name == "notebooks" else _ProgressPath.cwd()
_progress_sys.path.append(str(_PROGRESS_ROOT / "src"))
try:
    from election_network_progress import make_tracker
    progress = make_tracker("04_entity_extraction_candidate_dictionary", total_steps=8, root=_PROGRESS_ROOT)
except Exception as _progress_error:
    print(f"Progress tracker could not start: {_progress_error}")
    class _FallbackProgress:
        def __init__(self): self.current = 0
        def step(self, label):
            self.current += 1
            print(f"[stage {self.current}] {label}", flush=True)
        def info(self, label): print(f"[info] {label}", flush=True)
        def done(self, label="Notebook completed"): print(f"✓ {label}", flush=True)
    progress = _FallbackProgress()


## Purpose

This notebook extracts hashtags, mentions, domains, candidates, and topics from tweets. It also creates a candidate dictionary from the Senate results dataset, with manual variants for common nicknames and hashtag forms.

In [ ]:
progress.step("Purpose")
from pathlib import Path
import sys
import warnings
warnings.filterwarnings("ignore")

# Make ../src importable when running from notebooks/
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(ROOT / "src"))

from election_network_utils import *
paths = ensure_dirs(ROOT)

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_rows", 80)
px.defaults.template = "plotly_white"

RAW = paths["raw"]
PROCESSED = paths["processed"]
FIGURES = paths["figures"]
INTERACTIVE = paths["interactive"]
TABLES = paths["tables"]
NETWORKS = paths["networks"]
REPORT_ASSETS = paths["report_assets"]

print("Project root:", ROOT)


In [ ]:
progress.step("Purpose")
tweets = pd.read_csv(PROCESSED / "tweets_cleaned_base.csv", parse_dates=["createdAt"])
senate = pd.read_csv(RAW / "senate25-final_updated.csv", nrows=5)
candidate_ref = build_candidate_reference(senate)
candidate_dict = build_candidate_dictionary(candidate_ref)

candidate_ref.to_csv(PROCESSED / "candidate_reference.csv", index=False)
candidate_dict.to_csv(PROCESSED / "candidate_dictionary.csv", index=False)

print("Candidates:", len(candidate_ref))
print("Candidate variants:", len(candidate_dict))
display(candidate_dict.head(30))


In [ ]:
progress.step("Purpose")
# Candidate and topic extraction.
# This may take a few minutes on the full dataset because it scans 217k+ tweets.
# Keep this as one controlled extraction step so later notebooks run faster.

topic_dict = default_topic_dictionary()

tweets["hashtags"] = tweets["hashtags"].apply(parse_list_value) if "hashtags" in tweets.columns else tweets["text"].apply(extract_hashtags)
tweets["mentions"] = tweets["mentions"].apply(parse_list_value) if "mentions" in tweets.columns else tweets["text"].apply(extract_mentions)
tweets["domains"] = tweets["domains"].apply(parse_list_value) if "domains" in tweets.columns else tweets["text"].apply(extract_domains)

tweets["candidates_mentioned"] = tweets["text"].apply(lambda x: detect_candidates(x, candidate_dict))
tweets["topics_detected"] = tweets["text"].apply(lambda x: detect_topics(x, topic_dict))
tweets["candidate_count"] = tweets["candidates_mentioned"].apply(len)
tweets["topic_count"] = tweets["topics_detected"].apply(len)

tweets.to_csv(PROCESSED / "tweets_with_entities.csv", index=False)
print("Saved:", PROCESSED / "tweets_with_entities.csv")
display(tweets[["text", "hashtags", "mentions", "candidates_mentioned", "topics_detected"]].head(10))


In [ ]:
progress.step("Purpose")
# Frequency tables
hashtag_counts = Counter(h for tags in tweets["hashtags"] for h in tags)
mention_counts = Counter(m for mentions in tweets["mentions"] for m in mentions)
candidate_counts = Counter(c for cs in tweets["candidates_mentioned"] for c in cs)
topic_counts = Counter(t for ts in tweets["topics_detected"] for t in ts)

def counter_df(counter, name):
    return pd.DataFrame(counter.most_common(), columns=[name, "count"])

hashtag_df = counter_df(hashtag_counts, "hashtag")
mention_df = counter_df(mention_counts, "mention")
candidate_df = counter_df(candidate_counts, "candidate")
topic_df = counter_df(topic_counts, "topic")

hashtag_df.to_csv(PROCESSED / "hashtag_frequency.csv", index=False)
mention_df.to_csv(PROCESSED / "mention_frequency.csv", index=False)
candidate_df.to_csv(PROCESSED / "candidate_mention_frequency.csv", index=False)
topic_df.to_csv(PROCESSED / "topic_frequency.csv", index=False)

display(hashtag_df.head(20))
display(candidate_df.head(20))
display(topic_df.head(20))


In [ ]:
progress.step("Purpose")
fig = px.bar(candidate_df.head(25).sort_values("count"), x="count", y="candidate", orientation="h", title="Top detected candidates by Twitter/X mention count")
fig.update_layout(height=760, yaxis_title="", xaxis_title="Mentions")
save_plotly(fig, INTERACTIVE / "04_top_detected_candidates.html", FIGURES / "04_top_detected_candidates.png")
fig.show()


In [ ]:
progress.step("fig = px.bar(hashtag_df.head(25).sort_values('count'), x='count', y='hashtag', orientation='h', title='Top hashtags afte")
fig = px.bar(hashtag_df.head(25).sort_values("count"), x="count", y="hashtag", orientation="h", title="Top hashtags after extraction")
fig.update_layout(height=760, yaxis_title="", xaxis_title="Uses")
save_plotly(fig, INTERACTIVE / "04_top_hashtags.html", FIGURES / "04_top_hashtags.png")
fig.show()


In [ ]:
progress.step("fig = px.bar(topic_df.sort_values('count'), x='count', y='topic', orientation='h', title='Detected election topic freque")
fig = px.bar(topic_df.sort_values("count"), x="count", y="topic", orientation="h", title="Detected election topic frequency")
fig.update_layout(height=560, yaxis_title="", xaxis_title="Tweets")
save_plotly(fig, INTERACTIVE / "04_topic_frequency.html", FIGURES / "04_topic_frequency.png")
fig.show()


In [ ]:
progress.step("candidate_topic_edges = make_bipartite_edges(tweets['candidates_mentioned'], tweets['topics_detected'], left_name='candi")
# Candidate-topic heatmap
candidate_topic_edges = make_bipartite_edges(tweets["candidates_mentioned"], tweets["topics_detected"], left_name="candidate", right_name="topic")
candidate_topic_edges.to_csv(PROCESSED / "candidate_topic_edges.csv", index=False)

if not candidate_topic_edges.empty:
    top_candidates = candidate_df.head(20)["candidate"].tolist()
    pivot = candidate_topic_edges[candidate_topic_edges["candidate"].isin(top_candidates)].pivot_table(index="candidate", columns="topic", values="weight", fill_value=0)
    plt.figure(figsize=(16, 10))
    sns.heatmap(pivot, cmap="mako", linewidths=.5)
    plt.title("Candidate-topic association heatmap")
    plt.xlabel("Topic")
    plt.ylabel("Candidate")
    plt.tight_layout()
    plt.savefig(FIGURES / "04_candidate_topic_heatmap.png", dpi=220, bbox_inches="tight")
    plt.show()
else:
    print("No candidate-topic edges detected.")


## Run complete

The final cell closes the progress bar and confirms that all tracked notebook stages finished.


In [ ]:
progress.done("Notebook run completed")
